# Polars Part 1: Data Prep and Core Relationships

This smaller notebook covers loading with Polars, reshaping, merging, per-capita features, the Finland example, and the early time-series relationship views.

Links:
- [Polars notebook overview](README.md)
- [Shared helpers](../functions.py)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / 'data' / 'co2_data.csv').exists() and (candidate / 'notebooks' / 'functions.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
DATA_DIR = REPO_ROOT / 'data'

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
import umap.umap_ as umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display

import requests

from functions import (
    apply_correlation_to_df,
    normalize_column,
    classify_income_group,
    compute_energy_mix_shares,
    plot_dual_axis_timeseries,
    z_score_column,
    safe_divide,
    fetch_wikipedia_gni_table,
    fetch_world_bank_xml_records,
    run_kmeans_elbow,
    flatten_owid_panel_json,
    prepare_source_trend_table,
    compute_group_concentration,
    get_top_bottom_n,
)

sns.set_theme(style="whitegrid")
pd.options.display.float_format = '{:,.2f}'.format


## Setup

Import libraries and configure plot styling. We use `seaborn` for statistical graphics with a clean whitegrid theme, and `autoreload` to hot-reload helper functions from [functions.py](../functions.py) during development.

# Carbon Emissions and Economic Development: A Visual Analysis

The central question driving this analysis is whether economic growth necessarily comes at the cost of rising carbon emissions - or whether countries can **decouple** the two. This is one of the most consequential questions in climate policy: if decoupling is possible, it suggests that prosperity and sustainability are not mutually exclusive.

To investigate this, we combine two authoritative datasets:

1. **CO2 Emissions** (Our World in Data, 1750–2024) - historical emissions by country, measured in millions of tonnes
2. **GDP** (World Bank, 1960–2024) - economic output in current USD
3. **Electricity** (Our World in Data, 2000-2024) - Share of electricity production methods of the countries

By normalizing both metrics to per-capita values and computing Pearson correlations over time, we can classify countries along a spectrum: from those where growth and emissions move in lockstep (strong coupling) to those where GDP continues to rise while emissions fall (decoupling). We then ask whether this pattern is systematic - do high-income countries decouple more than low-income ones?

## CO2 Emissions Data

We load the Our World in Data CO2 dataset, which contains 79 columns spanning energy mix, land use, and emissions breakdowns. For this analysis we reduce it to five key variables: `country`, `year`, `iso_code`, `population`, and `co2` (total production-based CO2 emissions in millions of tonnes).

Two important filtering steps:
- **Drop rows without `iso_code`**: The dataset includes aggregate entities like "Africa", "OECD", and "World" that lack ISO country codes. Removing these ensures we work exclusively with individual nation-states.
- **Filter to post-1960**: GDP data from the World Bank only begins in 1960, so we align the time range to enable a clean merge later.

In [ ]:
co2_df = pl.read_csv(DATA_DIR / 'co2_data.csv')

# choose what columns to keep (can be changed, but this is the most important)
selected_columns = ['country', 'year', 'iso_code', 'population', 'co2']

# drop the columns that are not in the selected_columns list
co2_df = co2_df.select(selected_columns)

# remove entries with no iso code(Continents and other groups)
co2_df = (
    co2_df
    .with_columns(pl.col("co2")
                    .str.strip_chars()
                    .replace("", None)
                    .cast(pl.Float64))
    
    .filter(pl.col("iso_code") != "")
    .filter(pl.col("year") > 1960))


co2_df


## GDP Data

The World Bank GDP dataset arrives in **wide format** - one column per year (1960, 1961, ..., 2024). This is convenient for spreadsheet viewing but incompatible with tidy-data principles needed for plotting and merging. We use `pl.unipivot()` to reshape it into long format with one row per country-year observation.

Key steps:
- **Melt** year columns into `year` (int) and `gdp` (numeric) columns
- **Rename** `Country Code` to `iso_code` to create a shared merge key with the CO2 dataset
- **Drop missing GDP values** - not all countries have GDP records for every year, particularly in earlier decades or for newly independent states

In [ ]:
# load GDP data
gdp_df = pl.read_csv(DATA_DIR / 'gdp_data.csv')

# reshape from wide to long format
# melt the year columns into rows
year_columns = [col for col in gdp_df.columns if col.isdigit()]
gdp_df = gdp_df.unpivot(
    index=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    on=year_columns,
    variable_name='year',
    value_name='gdp')

# rename Country Code to iso_code for merging
gdp_df = gdp_df.rename({'Country Code': 'iso_code'})

# keep only the columns we need
gdp_df = gdp_df[['iso_code', 'year', 'gdp']]

gdp_df = (gdp_df
          .with_columns(pl.col("year").str.to_integer())
          .with_columns(pl.col("gdp").str.strip_chars().replace("", None).cast(pl.Float64))
          .drop_nulls(subset=["gdp"]))

gdp_df

## Merging the Datasets

We perform a **left join** of GDP onto the CO2 dataframe using `iso_code` and `year` as composite keys. A left join preserves every CO2 record and attaches GDP where available - countries or years without World Bank GDP data simply receive NaN. This is preferable to an inner join because it avoids silently discarding emission records that are still valuable for other analyses.

### Missing Data Heatmap

Before proceeding, we visualize data completeness. The heatmap below shows each variable as a column and each record as a row, with bright cells indicating missing values. This diagnostic is important because it reveals whether missingness is **random** or **systematic** - for instance, GDP data may be consistently absent for certain countries or time periods, which would bias any analysis that silently drops incomplete rows.

In [ ]:
# merge the datasets on iso_code and year
base_df = co2_df.join(
    gdp_df,
    on=['iso_code', 'year'],
    how='left',)

# missing data heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(base_df.select(pl.all().is_null()), yticklabels=False, cbar=False, cmap='magma_r', ax=ax)
ax.set_title("Missing Data Overview After Merge")

plt.tight_layout()
plt.show()

In [ ]:
# return null values per column
base_df.select(pl.all().null_count())

### Data Quality Summary

The GDP column shows the most missingness - this is expected since many countries (especially newly independent or conflict-affected states) lack World Bank GDP records in earlier decades. Importantly, the missingness is **systematic** rather than random: it concentrates in specific countries and time periods. This means any analysis involving GDP will implicitly exclude these observations, so conclusions are most robust for the subset of countries with consistent economic data.

## Per-Capita Metrics

Absolute CO2 and GDP figures are dominated by population size - China and India will always top the charts simply because they have the most people, not necessarily because their economies or industries are more carbon-intensive on a per-person basis. Dividing by population yields **per-capita** values that enable fairer cross-country comparisons: how much does the average citizen emit, and how wealthy is the average citizen?

- **CO2 per capita** is expressed in **tonnes per person** (the raw CO2 column is in millions of tonnes, so we multiply by 10⁶ before dividing by population).
- **GDP per capita** is in **current USD per person**.

We use `np.divide` with a `where` guard to handle zero or missing population entries without raising division errors.

In [ ]:
# CO2 is in millions of tonnes; multiply by 1e6 to get tonnes, then divide by population
base_df = base_df.with_columns(
    (pl.col("co2") / pl.col("population") * 1e6)
    .replace([float("inf"), -float("inf")], None)
    .alias("co2_per_capita"))

base_df = base_df.with_columns(
    (pl.col("gdp") / pl.col("population"))
    .replace([float("inf"), -float("inf")], None)
    .alias("gdp_per_capita"))

# Log-transform per-capita metrics for better visualization
base_df = base_df.with_columns(np.log1p(pl.col('gdp_per_capita'))
                               .alias("log_gdp_pc"))
base_df = base_df.with_columns(np.log1p(pl.col('co2_per_capita'))
                              .alias("log_co2_pc"))                     

# compare raw and log-transformed data as histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(base_df.select("gdp_per_capita"), bins=50, edgecolor='white')
axes[0].set_title('GDP per Capita – Raw Data')
axes[1].hist(base_df.select("log_gdp_pc"), bins=50, edgecolor='white', color='seagreen')
axes[1].set_title('GDP per Capita – Log-Transformed (approx. normal)')

plt.tight_layout()
plt.show()


# Analysis: Distribution and transformation of the GDP-per-capita

- **Raw-Data(Left)**: The distribution leans strongly left. Majority of the countries are in the lower income groups, while a few very rich countries form a sort of tail.

- **Log-Transformation (Right)**: By using log(x) the distribution becomes closer to the Gaussian Normal Distribution. That way we can weigh relative differences of the countries equally.


# Correlation Analysis

To quantify the relationship between economic growth and carbon emissions, we compute the **Pearson correlation coefficient** (r) between GDP per capita and CO2 per capita for each country across its full available time series.

- **r close to +1** indicates strong coupling - GDP and emissions rise together, typical of industrializing economies reliant on fossil fuels.
- **r close to 0** suggests no linear relationship - the two metrics move independently.
- **r < 0** indicates **decoupling** - GDP continues to grow while emissions decline, often driven by transitions to services-based economies, renewable energy adoption, or efficiency gains.

A minimum of 5 data points per country is required to compute a meaningful correlation; countries with fewer observations are excluded. It is worth noting that Pearson r captures only *linear* association - a country that industrialized heavily, peaked, and then decoupled will show a moderate positive r despite having a clear structural break in its emission trajectory.

In [ ]:
base_df = apply_correlation_to_df(base_df)

# Finland
country = 'Finland'
sel_df = base_df.filter(pl.col("country") == country)

# fit a linear trend line
x = sel_df["gdp_per_capita"].to_numpy()
y = sel_df["co2_per_capita"].to_numpy()

slope, intercept = np.polyfit(x, y, deg=1)
print(f"Trend: CO2_pc = {slope:.6f} × GDP_pc + {intercept:.2f}")

sel_df

## Country-Level Analysis: GDP vs CO2 Over Time

The dual-axis line plot below overlays GDP per capita and CO2 per capita for a selected country on a shared timeline. Two independent y-axes are necessary because the two metrics operate on vastly different scales (dollars vs. tonnes), but their *temporal trajectories* are directly comparable.

This visualization directly tests the core question of the analysis:
- **Parallel upward trends** indicate that the economy remains carbon-intensive - growth is fuelled by fossil energy.
- **Diverging trends** (GDP rising, CO2 flattening or declining) are evidence of **decoupling**, often driven by shifts toward service economies, renewable energy, or industrial efficiency.

The annotated Pearson r value provides a single-number summary of the visual relationship.

In [ ]:
plot_dual_axis_timeseries(
    df=sel_df,
    x_col='year',
    y1_col='gdp_per_capita',
    y2_col='co2_per_capita',
    y1_label='GDP per Capita (USD)',
    y2_label='CO2 per Capita (tonnes)',
    title=f'GDP per Capita vs CO2 Emissions in {country}',
    correlation_value=sel_df.select(pl.col("correlation").last()).item()
)

### Global Averages: CO2 per Capita vs GDP per Capita Over Time

Before zooming into individual countries, we first examine the **global average** trajectories. The dual-axis line plot below overlays the mean CO2 per capita (left axis, blue) and mean GDP per capita (right axis, orange) across all countries for each year.

This view reveals the macro-level tension: global GDP per capita has risen steadily since 1960, while average CO2 per capita shows signs of plateauing or declining in recent decades - suggesting that **aggregate decoupling** may already be underway at the global level.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

sns.lineplot(ax=ax1, x='year', y='co2_per_capita', data=base_df,
             color='steelblue', linewidth=2, label='CO2 per capita')
sns.lineplot(ax=ax2, x='year', y='gdp_per_capita', data=base_df,
             color='darkorange', linewidth=2, label='GDP per capita')

ax1.set_xlabel('Year')
ax1.set_ylabel('CO2 per capita (tonnes)', color='steelblue')
ax2.set_ylabel('GDP per capita (USD)', color='darkorange')
ax1.set_title('Global Average: CO2 per Capita vs GDP per Capita (1960-2024)')

# combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax2.get_legend().remove()

plt.tight_layout()
plt.show()